## linear operator base

In [ ]:
# from scipy.sparse import csr_matrix
# offsets = csr_matrix([[1, 0, 2], [0, -1, 0], [0, 0, 3]])
# print(offsets)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4 stored elements and shape (3, 3)>
  Coords	Values
  (0, 0)	1
  (0, 2)	2
  (1, 1)	-1
  (2, 2)	3


In [ ]:
# import numpy as np
# from scipy.sparse.linalg import LinearOperator
# def mv(v):
#     return np.array([2*v[0], 3*v[1]])

# A = LinearOperator((2,2), matvec=mv, shape=(2, 2))

# print(A.matvec(np.ones(2)))

# print(A*np.ones(2))

# print(A@np.ones(2))

[2. 3.]
[2. 3.]
[2. 3.]


In [ ]:
# from functools import reduce
# product = lambda c: reduce(lambda a, b: a * b, c)

## scalars computation

In [9]:
import torch
import os

while not os.getcwd().endswith("ScalarEMLP"):
    os.chdir("..")

from scalaremlp.nn.objax import comp_inner_products

# simulate trajectories of the data
# for instance
torch.manual_seed(26)
x=torch.rand((500,4,3))

scalar_no_sqrt=comp_inner_products(x,take_sqrt=False)

scalar_sqrt=comp_inner_products(x,take_sqrt=True)

print(scalar_no_sqrt.shape)
print(scalar_no_sqrt)
print(scalar_sqrt.shape)
print(scalar_sqrt)

(500, 16)
[[1.9305224  0.97942096 1.1065502  ... 0.62959826 0.8422622  1.3962443 ]
 [1.0277202  0.6632383  0.92761177 ... 0.8561057  0.48384613 1.2409244 ]
 [1.606065   0.75172716 1.1976011  ... 0.5952411  0.9182765  1.51133   ]
 ...
 [0.49498877 0.69599295 0.285209   ... 0.29812863 0.53418374 0.94372195]
 [0.26997972 0.55465925 0.60559297 ... 0.3868973  0.61641985 0.26532534]
 [0.9565554  0.7708498  0.6784344  ... 0.4729681  0.33971164 0.63850605]]
(500, 20)
[[1.3894324  0.8678878  0.96632785 ... 0.62959826 0.8422622  1.3962443 ]
 [1.0137653  1.1246741  0.91671187 ... 0.8561057  0.48384613 1.2409244 ]
 [1.2673062  0.8205871  1.0156397  ... 0.5952411  0.9182765  1.51133   ]
 ...
 [0.7035544  1.0219016  0.62781656 ... 0.29812863 0.53418374 0.94372195]
 [0.51959574 1.0903194  1.3273693  ... 0.3868973  0.61641985 0.26532534]
 [0.9780365  0.9690864  1.045203   ... 0.4729681  0.33971164 0.63850605]]


In [5]:
B1=x[0,:,:]
B2=x[1,:,:]
M1=B1@B1.T
M2=B2@B2.T
print(M1)
print(M2)

B=torch.stack((M1,M2))

tensor([[2.1623, 1.4237, 1.4731],
        [1.4237, 1.1633, 0.9831],
        [1.4731, 0.9831, 1.2687]])
tensor([[1.5195, 1.3161, 0.3171],
        [1.3161, 2.4052, 0.5338],
        [0.3171, 0.5338, 0.1442]])


In [6]:
G = torch.diag(-torch.ones(4))
G[0,0] = 1
print(G.unsqueeze(0))
print(x)
G = torch.einsum('bix,bxj->bij', x, G.unsqueeze(0))
# G = torch.einsum('cix,cxj->cij', x, G.unsqueeze(0))
print(G)

print()
scalars = torch.einsum('bij,bkj->bik', G, x)
print(scalars)

# simplified version: since the matrix is symmetric, take the upper trianglar part, flatten it
scalars = torch.triu(scalars).view(-1, 3**2)
# print(scalars)
# print(torch.nonzero(scalars[0]))
scalars = scalars[:, torch.nonzero(scalars[0]).squeeze(-1)]
print(scalars)

tensor([[[ 1.,  0.,  0.,  0.],
         [ 0., -1.,  0.,  0.],
         [ 0.,  0., -1.,  0.],
         [ 0.,  0.,  0., -1.]]])
tensor([[[0.6924, 0.9437, 0.2241, 0.8614],
         [0.4354, 0.4565, 0.5851, 0.6503],
         [0.0328, 0.8891, 0.2437, 0.6463]],

        [[0.7371, 0.9817, 0.1080, 0.0264],
         [0.6791, 0.7137, 0.8596, 0.8341],
         [0.1674, 0.1634, 0.2924, 0.0625]]])
tensor([[[ 0.6924, -0.9437, -0.2241, -0.8614],
         [ 0.4354, -0.4565, -0.5851, -0.6503],
         [ 0.0328, -0.8891, -0.2437, -0.6463]],

        [[ 0.7371, -0.9817, -0.1080, -0.0264],
         [ 0.6791, -0.7137, -0.8596, -0.8341],
         [ 0.1674, -0.1634, -0.2924, -0.0625]]])

tensor([[[-1.2034, -0.8207, -1.4278],
         [-0.8207, -0.7841, -0.9546],
         [-1.4278, -0.9546, -1.2666]],

        [[-0.4329, -0.3149, -0.0702],
         [-0.3149, -1.4828, -0.3064],
         [-0.0702, -0.3064, -0.0881]]])
tensor([[-1.2034, -0.8207, -1.4278, -0.7841, -0.9546, -1.2666],
        [-0.4329, -0.3149, -0

In [7]:
G = torch.diag(-torch.ones(4))
G[0,0] = 1

M1=x[0,:,:]@G@x[0,:,:].T
M2=x[1,:,:]@G@x[1,:,:].T

print(torch.stack((M1,M2)))

tensor([[[-1.2034, -0.8207, -1.4278],
         [-0.8207, -0.7841, -0.9546],
         [-1.4278, -0.9546, -1.2666]],

        [[-0.4329, -0.3149, -0.0702],
         [-0.3149, -1.4828, -0.3064],
         [-0.0702, -0.3064, -0.0881]]])


In [8]:
N=x.shape[0]
scalars = torch.einsum('bik,bjl->bijkl', x, x) #[N, n, n, dim, dim]
print(scalars.shape)
print(scalars)

torch.Size([2, 3, 3, 4, 4])
tensor([[[[[4.7946e-01, 6.5345e-01, 1.5515e-01, 5.9648e-01],
           [6.5345e-01, 8.9058e-01, 2.1145e-01, 8.1295e-01],
           [1.5515e-01, 2.1145e-01, 5.0204e-02, 1.9302e-01],
           [5.9648e-01, 8.1295e-01, 1.9302e-01, 7.4208e-01]],

          [[3.0151e-01, 3.1612e-01, 4.0512e-01, 4.5032e-01],
           [4.1092e-01, 4.3084e-01, 5.5214e-01, 6.1373e-01],
           [9.7564e-02, 1.0229e-01, 1.3109e-01, 1.4572e-01],
           [3.7510e-01, 3.9328e-01, 5.0400e-01, 5.6023e-01]],

          [[2.2682e-02, 6.1566e-01, 1.6876e-01, 4.4752e-01],
           [3.0913e-02, 8.3908e-01, 2.3001e-01, 6.0992e-01],
           [7.3395e-03, 1.9922e-01, 5.4610e-02, 1.4481e-01],
           [2.8218e-02, 7.6593e-01, 2.0996e-01, 5.5675e-01]]],


         [[[3.0151e-01, 4.1092e-01, 9.7564e-02, 3.7510e-01],
           [3.1612e-01, 4.3084e-01, 1.0229e-01, 3.9328e-01],
           [4.0512e-01, 5.5214e-01, 1.3109e-01, 5.0400e-01],
           [4.5032e-01, 6.1373e-01, 1.4572e-01, 5